# LSTM model COVID 19 - využitie neurónovej siete

### Príprava a spracovanie dát (Data Preprocessing)

Táto úvodná časť projektu sa zameriava na prípravu a spracovanie surových dát pre modelovanie. Ide o kritické kroky na zabezpečenie spoľahlivosti a presnosti následných predikcií.

Kód demonštruje tri kľúčové procesy:

1.  **Načítanie a filtrovanie dát:** Dáta o potvrdených prípadoch COVID-19 sa načítajú z verejne dostupného zdroja na GitHube. Následne sa pre cielenú analýzu dáta filtrujú tak, aby obsahovali iba záznamy týkajúce sa Slovenska.
2.  **Škálovanie (Scaling):** Dáta sa normalizujú na rozsah od 0 do 1. Tento krok je nevyhnutný pre efektívne trénovanie neurónovej siete, pretože to pomáha zabrániť chybe, kde niektoré dáta prevážia nad inými.
3.  **Tvorba sekvencií:** Dáta sa rozdelia do špecifických sekvencií a pripravia sa na vstup do modelu LSTM. Každá sekvencia (reprezentovaná premennou `trainX`) sa stane trénovacou vzorkou, z ktorej sa predpovedá nasledujúca hodnota (`trainY`).

Výsledkom tejto fázy je čistý a upravený súbor dát, pripravený na použitie v prediktívnych modeloch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf

# Krok 1: Načítanie a príprava dát z webu
url = 'https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv'

try:
    df = pd.read_csv(url, parse_dates=['date'])

    # Skontrolujeme, či sa dáta vôbec načítali
    if df.empty:
        print("Chyba: Dáta sa nenačítali z URL. DataFrame je prázdny.")
    else:
        print("Dáta sa úspešne načítali z URL.")

        # Filtrujeme dáta pre Slovensko
        df_country = df[df['location'] == 'Slovakia'].copy()

        # Ošetrenie chýbajúcich hodnôt a nastavenie indexu
        df_country.set_index('date', inplace=True)
        df_country.drop('location', axis=1, inplace=True)
        df_country.dropna(inplace=True, subset=['new_cases'])

        if df_country.empty:
            print("\nChyba: DataFrame df_country je prázdny po filtrovaní. Skontroluj, či názov 'Slovakia' sedí.")
        else:
            print(f"\nPočet záznamov pre Slovensko: {len(df_country)}")

            # Krok 2: Škálovanie dát
            data_lstm = df_country['new_cases'].values.reshape(-1, 1)
            scaler = MinMaxScaler(feature_range=(0, 1))
            scaled_data = scaler.fit_transform(data_lstm)

            # Krok 3: Vytvorenie sekvencií pre LSTM
            look_back = 30
            def create_dataset(dataset, look_back=1):
                dataX, dataY = [], []
                for i in range(len(dataset) - look_back):
                    a = dataset[i:(i + look_back), 0]
                    dataX.append(a)
                    dataY.append(dataset[i + look_back, 0])
                return np.array(dataX), np.array(dataY)

            train_size = int(len(scaled_data) * 0.8)
            train, test = scaled_data[0:train_size, :], scaled_data[train_size:len(scaled_data), :]

            trainX, trainY = create_dataset(train, look_back)
            testX, testY = create_dataset(test, look_back)

            trainX = np.reshape(trainX, (trainX.shape[0], trainX.shape[1], 1))
            testX = np.reshape(testX, (testX.shape[0], testX.shape[1], 1))

            print("\n--- Výsledky prípravy dát pre LSTM ---")
            print("Tvar trénovacej sady (X):", trainX.shape)
            print("Tvar trénovacej sady (Y):", trainY.shape)

except Exception as e:
    print(f"Nastala chyba pri načítaní dát: {e}")

# Modelovanie s neurónovými sieťami: LSTM
Po preskúmaní štatistických modelov je čas posunúť sa k pokročilejším metódam. V tejto časti projektu som sa zamerala na modelovanie s rekurentnými neurónovými sieťami, konkrétne s modelom LSTM (Long Short-Term Memory).

Na rozdiel od štatistických modelov dokážu LSTM siete „pamätať si“ dlhodobé vzory a závislosti v dátach, vďaka čomu sú ideálne na predikciu komplexných časových radov.

Nasledujúci kód demonštruje tri kľúčové kroky:

Tvorba modelu: Vytvorila som sekvenčný model, ktorý obsahuje jednu LSTM vrstvu na spracovanie časových údajov a výstupnú vrstvu, ktorá generuje predikciu.

Kompilácia modelu: V tejto fáze bol model nakonfigurovaný s optimalizátorom adam, ktorý sa stará o efektívne trénovanie, a s metrikou mean_squared_error, ktorá meria presnosť predikcií.

Trénovanie modelu: Model bol trénovaný na upravených dátach, pričom som mu dala 100 epoch. Počas trénovania sa model učí z dát, aby dokázal čo najpresnejšie predpovedať vývoj.

Na konci je zobrazený súhrn modelu, ktorý poskytuje prehľad o jeho architektúre a počte trénovateľných parametrov.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# Predpokladáme, že 'trainX' a 'trainY' sú už definované
# Ak nie, spusti najprv predchádzajúci kód

# Tvorba LSTM modelu
model = Sequential()
model.add(LSTM(50, input_shape=(trainX.shape[1], 1))) # Vrstva LSTM s 50 neurónmi
model.add(Dense(1)) # Výstupná vrstva, ktorá predikuje jeden číselný výsledok

# Kompilácia modelu
model.compile(loss='mean_squared_error', optimizer='adam')

# Trénovanie modelu
# Epochs: počet prechodov cez celý dataset
# Batch_size: počet vzoriek v jednej dávke, po ktorej sa aktualizujú váhy
history = model.fit(
    trainX,
    trainY,
    epochs=100,
    batch_size=1,
    verbose=2
)

# Zobrazenie súhrnu modelu
print("\n--- Súhrn LSTM modelu ---")
model.summary()

# Vyhodnotenie a vizualizácia predikcií modelu LSTM
Po natrénovaní modelu je nevyhnutné vizuálne aj numericky posúdiť jeho presnosť. V tejto časti projektu som sa zamerala na porovnanie predikovaných hodnôt s reálnym vývojom.

Nasledujúci kód demonštruje tieto kroky:

Predikcia a de-normalizácia: Natrénovaný model LSTM bol použitý na predpovedanie hodnôt z testovacej sady. Následne boli predikované hodnoty de-normalizované, aby sme ich vrátili na pôvodnú škálu a mohli ich porovnať s reálnymi hodnotami.

Vizualizácia výsledkov: Graficky som znázornila porovnanie reálnych (zelená čiara) a predikovaných (červená čiara) hodnôt. Táto vizualizácia poskytuje okamžitý prehľad o tom, ako presne sa model prispôsobil skutočnému vývoju.

Numerické metriky: Na kvantitatívne posúdenie presnosti modelu som použila dve dôležité metriky:

Mean Absolute Error (MAE): Priemerná absolútna odchýlka medzi predpoveďou a skutočnou hodnotou.

Root Mean Squared Error (RMSE): Podobne ako MAE, meria chybu modelu, ale silnejšie penalizuje väčšie odchýlky.

Tento proces vyhodnotenia je kľúčový na overenie spoľahlivosti modelu a slúži ako podklad pre porovnanie s inými prístupmi, napríklad so štatistickými modelmi ARIMA.

In [ ]:


from sklearn.metrics import mean_absolute_error, mean_squared_error

# Uistenie, že máme potrebné dáta
# testX, testY, scaler, df_country

# 1. Predikcia na testovacích dátach
predicted_lstm = model.predict(testX)

# 2. Invertovanie škálovania
# Predikcie sú teraz v rozsahu od 0 do 1, musíme ich vrátiť na pôvodnú škálu
predicted_lstm = scaler.inverse_transform(predicted_lstm)
testY_unscaled = scaler.inverse_transform(testY.reshape(-1, 1))

# 3. Vizuálne porovnanie predikcií
plt.figure(figsize=(15, 6))
plt.plot(df_country.index[len(df_country)-len(predicted_lstm):], testY_unscaled, label='Skutočné hodnoty', color='green')
plt.plot(df_country.index[len(df_country)-len(predicted_lstm):], predicted_lstm, label='LSTM predikcie', color='red', linestyle='--')
plt.title('Predikcie LSTM modelu vs. reálne hodnoty')
plt.xlabel('Dátum')
plt.ylabel('Počet nových prípadov')
plt.legend()
plt.show()

# 4. Výpočet MAE a RMSE
mae_lstm = mean_absolute_error(testY_unscaled, predicted_lstm)
rmse_lstm = np.sqrt(mean_squared_error(testY_unscaled, predicted_lstm))

print(f"Mean Absolute Error (MAE) pre LSTM: {mae_lstm:.2f}")
print(f"Root Mean Squared Error (RMSE) pre LSTM: {rmse_lstm:.2f}")

# Uloženie modelu a vytvorenie Gradio aplikácie

In [ ]:
# Uloženie natrénovaného modelu
model.save('lstm_model.keras')

In [ ]:
!pip install gradio
import gradio as gr
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Načítanie uloženého modelu
model = tf.keras.models.load_model('lstm_model.keras')

# Znova inicializujeme scaler, aby sme mohli dáta škálovať a odškálovať
scaler = MinMaxScaler(feature_range=(0, 1))
# Musíme natrénovať scaler na pôvodných dátach, aby sme ho mohli použiť pre predikciu
# (pretože nevieme, aké dáta dostaneme na vstupe)
df_country = pd.read_csv('https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv', parse_dates=['date'])
df_country = df_country[df_country['location'] == 'Slovakia'].copy()
data_lstm = df_country['new_cases'].values.reshape(-1, 1)
scaler.fit(data_lstm)

def predict_lstm(input_values):
    # Konvertovanie vstupu na numpy array
    try:
        input_array = np.array(input_values.split(',')).astype(float)

        # Kontrola, či je počet vstupov správny
        if len(input_array) != 30:
            return "Chyba: Zadaj presne 30 hodnôt oddelených čiarkou."

        # Zmena tvaru a škálovanie vstupu
        input_scaled = scaler.transform(input_array.reshape(-1, 1))
        input_lstm = np.reshape(input_scaled, (1, 30, 1))

        # Predikcia
        prediction_scaled = model.predict(input_lstm)

        # Odškálovanie predikcie na pôvodnú škálu
        prediction = scaler.inverse_transform(prediction_scaled)

        return f"Predikovaný počet prípadov na ďalší deň je: {int(np.maximum(0, prediction[0][0]))}"

    except Exception as e:
        return f"Chyba: Zadaj prosím platné číselné hodnoty. ({e})"

# Vytvorenie a spustenie Gradio rozhrania
iface = gr.Interface(
    fn=predict_lstm,
    inputs=gr.Textbox(lines=5, label="Zadaj posledných 30 hodnôt oddelených čiarkou"),
    outputs="text",
    title="Prediktor nových prípadov COVID-19 (LSTM)",
    description="Zadaj posledných 30 hodnôt nových prípadov COVID-19 a model predikuje počet prípadov na nasledujúci deň."
)

iface.launch()

# Porovnanie modelov ARIMA a LSTM
Na základe poskytnutých metrík je zrejmé, že model LSTM dosiahol v predikciách výrazne lepšie výsledky ako model SARIMA. Nižšie uvedená tabuľka jasne ukazuje, aký je rozdiel medzi týmito dvoma prístupmi.

Metrika	Model SARIMA	Model LSTM
Mean Absolute Error (MAE)	180.89	67.78
Root Mean Squared Error (RMSE)	291.31	109.12


MAE (Mean Absolute Error) predstavuje priemernú absolútnu odchýlku medzi predpovedanými a skutočnými hodnotami. Čím nižšia je táto hodnota, tým je model presnejší.

RMSE (Root Mean Squared Error) je podobná metrika, ale silnejšie penalizuje väčšie chyby v predikcii. Podobne ako pri MAE, aj tu platí, že nižšia hodnota znamená lepšiu presnosť.

**Záverečné hodnotenie**

Výsledky ukazujú, že model LSTM bol oveľa presnejší pri predpovedi nových prípadov. Je to preto, že LSTM siete, ako typ neurónových sietí, sú navrhnuté tak, aby dokázali zachytiť komplexné a nelineárne vzory v dátach, ktoré tradičné štatistické modely ako SARIMA nemusia byť schopné identifikovať. Vzhľadom na povahu dát (vývoj pandémie) sa zložitejšie modely, ktoré dokážu nájsť hlbšie závislosti, osvedčili ako účinnejšie.

# Záver

Tento projekt slúži ako praktická demonštrácia, že kombináciou rôznych prístupov (štatistických a ML modelov) môžeme dosiahnuť robustnejší a presnejší výsledok, ktorý je prispôsobený konkrétnym potrebám analýzy.

**Ďalšie odporúčania a možnosti rozšírenia projektu:**
Tento projekt je možné rozšíriť o ďalšie kroky, ktoré by mohli zlepšiť presnosť predikcií:

**Viac vstupných premenných:** Okrem počtu prípadov by bolo možné pridať ďalšie premenné, ako sú napríklad opatrenia vlády, dátum zavedenia karantény, alebo iné relevantné dáta.

**Optimalizácia parametrov:** Použitie pokročilých techník ako je Grid Search alebo Random Search na nájdenie optimálnych parametrov pre modely ARIMA aj LSTM.

**Priebeh projektu:** Celý projekt je dostupný na mojom GitHub profile, kde si môžete prezrieť a spustiť celý kód, ktorý slúži ako ukážka mojich technických zručností v oblasti dátovej analýzy a strojového učenia.